## 0. Imports and Setup

In [ ]:
import sys
import importlib
import logging
from pathlib import Path

import pandas as pd

# VS Code Jupyter sets CWD to the project root, so '.' is the right path.
PROJECT_ROOT = str(Path('.').resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Force reload so source changes are picked up without a kernel restart.
import src.data.injury_loader as _il_mod
importlib.reload(_il_mod)

from src.data.injury_loader import (
    fetch_il_transactions,
    parse_injury_type,
    compute_days_lost,
    build_injury_database,
    load_injury_database,
)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('notebook')
print('injury_loader loaded from:', _il_mod.__file__)

## 1. Configuration

In [ ]:
START_YEAR = 2015
END_YEAR   = 2024

# Flip to False for the full 2015–2024 pull.
TEST_MODE  = False
TEST_YEAR  = 2023

INJURIES_DIR = Path('data/raw/injuries')
INJURIES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Output dir : {INJURIES_DIR.resolve()}')
print(f'Test mode  : {TEST_MODE}')

## 2. Load Pitcher ID Filter

The transactions API returns every player — position players, coaches, two-way
players. We restrict to pitchers using the MLBAM IDs collected in notebook 01.
Any player who ever appeared as `pitcher` in Statcast is included.

In [ ]:
meta_path = Path('data/raw/player_metadata/pitchers.parquet')
if not meta_path.exists():
    raise FileNotFoundError(
        'Run notebook 01 first to generate data/raw/player_metadata/pitchers.parquet'
    )

pitcher_meta = pd.read_parquet(meta_path)
pitcher_ids  = set(pitcher_meta['player_id'].dropna().astype(int))
print(f'Pitcher filter: {len(pitcher_ids):,} unique MLBAM IDs')

## 3. Fetch Raw IL Transactions

Queries the MLB Stats API one month at a time. Each row is a single transaction
event — either a placement (IL) or an activation (IA). We keep both; pairing
happens in section 5.

In [ ]:
if TEST_MODE:
    il_raw = fetch_il_transactions(TEST_YEAR, TEST_YEAR)
else:
    il_raw = fetch_il_transactions(START_YEAR, END_YEAR)

print(f'\nTotal IL transaction records : {len(il_raw):,}')
print(f'Transaction type breakdown:')
print(il_raw['transaction_type'].value_counts().to_string())
print(f'\nDate range: {il_raw["transaction_date"].min().date()} → {il_raw["transaction_date"].max().date()}')
display(il_raw.head(8))

## 4. Filter to Pitchers

In [ ]:
before = len(il_raw)
il_pitchers = il_raw[il_raw['player_id'].isin(pitcher_ids)].copy()
after  = len(il_pitchers)

print(f'All players : {before:,} transactions')
print(f'Pitchers    : {after:,} transactions  ({after/before:.1%} of total)')
print(f'Unique pitchers with IL activity: {il_pitchers["player_id"].nunique():,}')

## 5. Inspect Raw Descriptions

Before parsing, look at real description strings to confirm the keyword patterns
will work correctly.

In [ ]:
placements_raw = il_pitchers[il_pitchers['transaction_type'] == 'IL'].copy()

print(f'Pitcher IL placements: {len(placements_raw):,}')
print('\nSample descriptions:')
for desc in placements_raw['description'].dropna().sample(min(10, len(placements_raw)), random_state=42):
    print(f'  {desc}')

## 6. Parse Injury Types

`parse_injury_type` applies keyword patterns in priority order. The first matching
pattern wins — elbow is checked before shoulder, shoulder before forearm, etc. —
so `"right elbow strain"` maps to `elbow`, not `other`.

In [ ]:
# Verify the parser on a few hand-crafted examples before applying at scale.
TEST_DESCRIPTIONS = [
    ("Placed on 15-day IL. Right elbow inflammation.",              "elbow"),
    ("Transferred to 60-day IL. Left shoulder rotator cuff tear.",  "shoulder"),
    ("Placed on 15-day IL. Right forearm strain.",                  "forearm"),
    ("Placed on 15-day IL. Right oblique strain.",                  "oblique"),
    ("Placed on 10-day IL. Left hamstring tightness.",              "hamstring"),
    ("Placed on 15-day IL. Right knee inflammation.",               "knee"),
    ("Placed on 10-day IL. Right hand blister.",                    "finger_hand"),
    ("Placed on 10-day IL. Illness.",                               "illness"),
    ("Placed on 15-day IL. Lower back tightness.",                  "back"),
]

print('Parser unit tests:')
all_pass = True
for desc, expected in TEST_DESCRIPTIONS:
    result = parse_injury_type(desc)
    status = '✓' if result == expected else '✗'
    if result != expected:
        all_pass = False
    print(f'  {status}  expected={expected:<12}  got={result:<12}  |  {desc[:60]}')

print(f'\nAll tests passed: {all_pass}')

In [ ]:
# Apply to the full pitcher placement dataset
placements_raw['injury_type'] = placements_raw['description'].apply(parse_injury_type)

print('Injury type distribution (placements only):')
counts = placements_raw['injury_type'].value_counts()
pct    = (counts / counts.sum() * 100).round(1)
display(pd.DataFrame({'count': counts, 'pct': pct}))

## 7. Pair Placements with Activations → Compute Days Lost

Each IL placement (typeCode `IL`) is matched to the earliest subsequent
activation (typeCode `IA`) for the same player. IL transfers (ITD — e.g.,
15-day to 60-day) are ignored for pairing because they extend an existing
stint rather than starting a new one. `days_lost` therefore spans the full
original placement to final activation, regardless of IL list changes in between.

Placements with no matching activation in any season are flagged `season_ending=True`
with `days_lost=None`.

In [ ]:
injury_db = compute_days_lost(il_pitchers)

print(f'IL stints (placements paired): {len(injury_db):,}')
print(f'  With activation date  : {injury_db["activation_date"].notna().sum():,}')
print(f'  Season-ending (no act): {injury_db["season_ending"].sum():,}')
print(f'\ndays_lost distribution (non-season-ending):')
print(injury_db[injury_db["days_lost"].notna()]["days_lost"].describe().round(1).to_string())

display(injury_db.head(8))

## 8. Validation

### 8a. Known-Injury Spot Checks

Verify specific well-documented 2023 IL stints appear correctly in the database.

In [ ]:
# Spot-check well-documented 2023 IL stints.
SPOT_CHECKS = {
    434378: 'Justin Verlander',
    477132: 'Clayton Kershaw',
    605483: 'Sandy Alcantara',
}

for pid, name in SPOT_CHECKS.items():
    rows = injury_db[injury_db['player_id'] == pid]
    if len(rows) == 0:
        print(f'{name} ({pid}): no IL stints found in {TEST_YEAR if TEST_MODE else str(START_YEAR)+"-"+str(END_YEAR)}')
    else:
        print(f'{name} ({pid}): {len(rows)} stint(s)')
        print(rows[['transaction_date','activation_date','days_lost','injury_type','description']]
              .to_string(index=False))
    print()

### 8b. Days Lost Sanity Checks

In [ ]:
paired = injury_db[injury_db['days_lost'].notna()].copy()

# Stints shorter than 10 days are suspicious — minimum IL is 10 days.
too_short = paired[paired['days_lost'] < 10]
print(f'Stints < 10 days: {len(too_short)}')
if len(too_short) > 0:
    display(too_short[['player_name','transaction_date','activation_date','days_lost','description']].head(5))

# Stints over 365 days are likely data errors (wrong-season activation matched).
too_long = paired[paired['days_lost'] > 365]
print(f'Stints > 365 days: {len(too_long)}')
if len(too_long) > 0:
    display(too_long[['player_name','transaction_date','activation_date','days_lost','description']].head(5))

print(f'\nMedian days lost: {paired["days_lost"].median():.0f}')
print(f'Mean days lost  : {paired["days_lost"].mean():.0f}')

### 8c. Injury Type Distribution

In [ ]:
print('Injury type distribution (all stints):')
counts = injury_db['injury_type'].value_counts()
pct    = (counts / counts.sum() * 100).round(1)
display(pd.DataFrame({'count': counts, 'pct_%': pct}))

print('\nMean days lost by injury type:')
print(
    injury_db[injury_db['days_lost'].notna()]
    .groupby('injury_type')['days_lost']
    .agg(['mean','median','count'])
    .round(1)
    .sort_values('mean', ascending=False)
    .to_string()
)

## 9. Save

In [ ]:
save_path = INJURIES_DIR / 'injury_database.parquet'
injury_db.to_parquet(save_path, index=False)
print(f'Saved {len(injury_db):,} IL stints → {save_path.resolve()}')

# Quick reload check
check = load_injury_database(str(save_path))
assert len(check) == len(injury_db), 'Row count mismatch on reload'
print('Reload check passed.')